In [ ]:
# Establish database connection

In [5]:
"""
Create the MySQL schema + tables from your SQL DDL using Python.

Requirements:
  pip install mysql-connector-python

Usage:
  1) Fill in MYSQL_HOST / MYSQL_USER / MYSQL_PASSWORD
  2) Run this script: python create_mydb.py
"""

import mysql.connector
from mysql.connector import errorcode


MYSQL_HOST = "localhost"
MYSQL_USER = "root"
MYSQL_PASSWORD = "realitygoon13"
DB_NAME = "mydb"

In [6]:
DDL_STATEMENTS = [
    # --- Preserve/disable checks like your script ---
    "SET @OLD_UNIQUE_CHECKS=@@UNIQUE_CHECKS, UNIQUE_CHECKS=0;",
    "SET @OLD_FOREIGN_KEY_CHECKS=@@FOREIGN_KEY_CHECKS, FOREIGN_KEY_CHECKS=0;",
    "SET @OLD_SQL_MODE=@@SQL_MODE, SQL_MODE='ONLY_FULL_GROUP_BY,STRICT_TRANS_TABLES,NO_ZERO_IN_DATE,NO_ZERO_DATE,ERROR_FOR_DIVISION_BY_ZERO,NO_ENGINE_SUBSTITUTION';",

    # --- Schema ---
    "CREATE SCHEMA IF NOT EXISTS `mydb` DEFAULT CHARACTER SET utf8;",
    "USE `mydb` ;",

    # --- Table: ensembl_genes ---
    """
    CREATE TABLE IF NOT EXISTS `mydb`.`ensembl_genes` (
      `ensembl_id` INT NOT NULL,
      `omim_id` INT NULL,
      `chromosome_location` INT NULL,
      `gene_start_loc` INT NULL,
      `gene_end_loc` INT NULL,
      `genome_version` VARCHAR(45) NULL,
      PRIMARY KEY (`ensembl_id`)
    ) ENGINE = InnoDB;
    """,

    # --- Table: Hgnc_genes ---
    """
    CREATE TABLE IF NOT EXISTS `mydb`.`Hgnc_genes` (
      `hgnc_id` INT NOT NULL,
      `gene_name` VARCHAR(45) NULL,
      `gene_aliases` VARCHAR(45) NULL,
      `omim_id` INT NULL,
      `ensembl_id` VARCHAR(45) NULL,
      `ensembl_genes_ensembl_id` INT NOT NULL,
      PRIMARY KEY (`hgnc_id`),
      INDEX `fk_Hgnc_genes_ensembl_genes_idx` (`ensembl_genes_ensembl_id` ASC) VISIBLE,
      CONSTRAINT `fk_Hgnc_genes_ensembl_genes`
        FOREIGN KEY (`ensembl_genes_ensembl_id`)
        REFERENCES `mydb`.`ensembl_genes` (`ensembl_id`)
        ON DELETE NO ACTION
        ON UPDATE NO ACTION
    ) ENGINE = InnoDB;
    """,

    # --- Table: OSL_diseases ---
    """
    CREATE TABLE IF NOT EXISTS `mydb`.`OSL_diseases` (
      `omim_id` INT NOT NULL,
      `disease_name` VARCHAR(45) NULL,
      `ensembl_id` VARCHAR(45) NULL,
      `ensembl_genes_ensembl_id` INT NOT NULL,
      PRIMARY KEY (`omim_id`),
      INDEX `fk_OSL_diseases_ensembl_genes1_idx` (`ensembl_genes_ensembl_id` ASC) VISIBLE,
      CONSTRAINT `fk_OSL_diseases_ensembl_genes1`
        FOREIGN KEY (`ensembl_genes_ensembl_id`)
        REFERENCES `mydb`.`ensembl_genes` (`ensembl_id`)
        ON DELETE NO ACTION
        ON UPDATE NO ACTION
    ) ENGINE = InnoDB;
    """,

    # --- Restore checks ---
    "SET SQL_MODE=@OLD_SQL_MODE;",
    "SET FOREIGN_KEY_CHECKS=@OLD_FOREIGN_KEY_CHECKS;",
    "SET UNIQUE_CHECKS=@OLD_UNIQUE_CHECKS;",
]


In [7]:

def run_ddl():
    # Connect without specifying the database first (schema might not exist yet)
    cnx = mysql.connector.connect(
        host=MYSQL_HOST,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        autocommit=False,
    )

    try:
        cur = cnx.cursor()
        for stmt in DDL_STATEMENTS:
            s = stmt.strip()
            if not s:
                continue
            try:
                cur.execute(s)
            except mysql.connector.Error as err:
                # Show statement context for easier debugging
                print("\nFAILED STATEMENT:\n", s)
                raise

        cnx.commit()
        print("✅ Schema and tables created successfully in database:", DB_NAME)

    finally:
        try:
            cur.close()
        except Exception:
            pass
        cnx.close()


if __name__ == "__main__":
    run_ddl()


✅ Schema and tables created successfully in database: mydb
